In [2]:
!pip install pyspark

In [3]:
import pyspark

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkAssignment") \
    .getOrCreate()

spark

In [34]:
url = "/content/Titanic-Dataset.csv"

df = spark.read.csv(url, header=True, inferSchema=True)

df

DataFrame[PassengerId: int, Survived: int, Pclass: int, Name: string, Sex: string, Age: double, SibSp: int, Parch: int, Ticket: string, Fare: double, Cabin: string, Embarked: string]

In [35]:
df.printSchema()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



In [36]:
df.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+------

In [37]:
df.count()

891

In [38]:
df.describe().show()

+-------+-----------------+-------------------+------------------+--------------------+------+------------------+------------------+-------------------+------------------+-----------------+-----+--------+
|summary|      PassengerId|           Survived|            Pclass|                Name|   Sex|               Age|             SibSp|              Parch|            Ticket|             Fare|Cabin|Embarked|
+-------+-----------------+-------------------+------------------+--------------------+------+------------------+------------------+-------------------+------------------+-----------------+-----+--------+
|  count|              891|                891|               891|                 891|   891|               714|               891|                891|               891|              891|  204|     889|
|   mean|            446.0| 0.3838383838383838| 2.308641975308642|                NULL|  NULL| 29.69911764705882|0.5230078563411896|0.38159371492704824|260318.54916792738| 32.20420

In [41]:
df.filter(df.Age > 30).show(5)
df.filter((df.Sex == 'female') & (df.Survived == 1)).show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+--------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|  Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+--------+-------+-----+--------+
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|PC 17599|71.2833|  C85|       C|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|  113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|  373450|   8.05| NULL|       S|
|          7|       0|     1|McCarthy, Mr. Tim...|  male|54.0|    0|    0|   17463|51.8625|  E46|       S|
|         12|       1|     1|Bonnell, Miss. El...|female|58.0|    0|    0|  113783|  26.55| C103|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+--------+-------+-----+--------+
only showing top 5 rows
+-----------+

In [42]:
#Count passengers by class
print("Count of passengers by Pclass:")
df.groupBy("Pclass").count().show()

#Average age by class
print("Average age of passengers by Pclass:")
df.groupBy("Pclass").avg("Age").show()

#Average fare by survival
print("Average fare by survival status (0 = not survived, 1 = survived):")
df.groupBy("Survived").avg("Fare").show()

Count of passengers by Pclass:
+------+-----+
|Pclass|count|
+------+-----+
|     1|  216|
|     3|  491|
|     2|  184|
+------+-----+

Average age of passengers by Pclass:
+------+------------------+
|Pclass|          avg(Age)|
+------+------------------+
|     1|38.233440860215055|
|     3| 25.14061971830986|
|     2| 29.87763005780347|
+------+------------------+

Average fare by survival status (0 = not survived, 1 = survived):
+--------+------------------+
|Survived|         avg(Fare)|
+--------+------------------+
|       1| 48.39540760233917|
|       0|22.117886885245877|
+--------+------------------+



In [45]:
import pandas as pd
import time

#Pandas
start = time.time()
pdf = pd.read_csv("Titanic-Dataset.csv")
print("Pandas load time:", time.time() - start)

start = time.time()
pdf[pdf.Age > 30]  # Filter example
print("Pandas filter time:", time.time() - start)

#Spark
start = time.time()
df.filter(df.Age > 30).count()  # Trigger action
print("Spark filter time:", time.time() - start)

start = time.time()
df.groupBy("Pclass").count().collect()  # Trigger aggregation
print("Spark groupBy count time:", time.time() - start)

Pandas load time: 0.015533685684204102
Pandas filter time: 0.016290664672851562
Spark filter time: 0.24709606170654297
Spark groupBy count time: 0.3740546703338623


Pandas is faster on a small dataset (~0.016 s) than Spark took longer (~0.25–0.37 s).
Spark works best with large datasets.